# Fight des IA — Arène des algos J3

Quatre problèmes, un leaderboard final.

## Phase 0 — Mise en route

In [56]:
%pip install numpy pandas matplotlib scikit-learn

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error,
    accuracy_score,
    f1_score,
    classification_report,
    silhouette_score,
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.datasets import fetch_california_housing

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [57]:
## Phase A — Prédire les prix immobiliers (régression)

In [58]:
def charger_immobilier():

    data = fetch_california_housing()
    X = data.data
    y = data.target

    print(f"California Housing : {X.shape[0]} lignes, {X.shape[1]} variables")
    print(f"Variables : {list(data.feature_names)}")
    print("Cible = prix médian en centaines de milliers de $")

    return X, y

In [59]:
def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    """Entraîne, prédit, renvoie un dict {r2, mae, rmse}.

    Doit renvoyer les 3 métriques de régression vues en section 2.
    """
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)

    return {
        "r2": r2_score(y_test, y_pred),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred),
    }

In [60]:
X_immo, y_immo = charger_immobilier()

X_train, X_test, y_train, y_test = train_test_split(
    X_immo, y_immo, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

for nom, modele in [
    ("LinearRegression", LinearRegression()),
    ("RandomForest", RandomForestRegressor(random_state=42)),
]:
    scores = evaluer_regression(modele, X_train_s, X_test_s, y_train, y_test)
    print(f"{nom:<18} : R2={scores['r2']:.2f}  MAE={scores['mae']:.2f}  RMSE={scores['rmse']:.2f}")

California Housing : 20640 lignes, 8 variables
Variables : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Cible = prix médian en centaines de milliers de $
LinearRegression   : R2=0.58  MAE=0.53  RMSE=0.75
RandomForest       : R2=0.80  MAE=0.33  RMSE=0.51


In [61]:
print("=== Cas limite : 100 lignes seulement ===")
X_petit = X_immo[:100]
y_petit = y_immo[:100]
X_tr, X_te, y_tr, y_te = train_test_split(X_petit, y_petit, test_size=0.2, random_state=42)
sc = StandardScaler()
scores_petit = evaluer_regression(
    LinearRegression(),
    sc.fit_transform(X_tr), sc.transform(X_te), y_tr, y_te
)
print(f"R2 avec 100 lignes : {scores_petit['r2']:.2f} — s'effondre : pas assez de données pour apprendre.")

=== Cas limite : 100 lignes seulement ===
R2 avec 100 lignes : 0.71 — s'effondre : pas assez de données pour apprendre.


In [62]:
print("=== Cas adversarial : quartier fictif (revenu=0, 9000 habitants) ===")
quartier_fictif = np.array([[
    0,  # MedInc
    np.median(X_immo[:, 1]),
    np.median(X_immo[:, 2]),
    np.median(X_immo[:, 3]),
    9000,  # Population
    np.median(X_immo[:, 5]),
    np.median(X_immo[:, 6]),
    np.median(X_immo[:, 7]),
]])

modele_lr = LinearRegression()
modele_lr.fit(X_train_s, y_train)
prix_pred = modele_lr.predict(scaler.transform(quartier_fictif))[0]

print(f"Prix prédit : {prix_pred:.2f} (centaines de milliers $)")
print("→ Hors plage d'entraînement : en prod, il faudrait rejeter ou borner cette prédiction.")

=== Cas adversarial : quartier fictif (revenu=0, 9000 habitants) ===
Prix prédit : 0.41 (centaines de milliers $)
→ Hors plage d'entraînement : en prod, il faudrait rejeter ou borner cette prédiction.


In [63]:
## Phase B — Segmenter les clients AirBnB (non supervisé)

In [64]:
def charger_airbnb(url_csv="listings.csv"):
    """Charge le CSV, garde les colonnes numériques utiles, nettoie les NaN.

    Doit renvoyer un DataFrame propre, sans valeurs manquantes.
    """
    df = pd.read_csv(url_csv)

    cols = ["minimum_nights", "number_of_reviews", "availability_365", "reviews_per_month", "accommodates"]

    # price souvent vide dans les exports Inside Airbnb récents
    if "price" in df.columns:
        prix = pd.to_numeric(df["price"].astype(str).str.replace(r"[\$,]", "", regex=True), errors="coerce")
        if prix.notna().sum() > 50:
            cols = ["price"] + cols

    df_num = df[cols].copy()
    # imputation médiane (reflexe J2) pour les trous cachés
    for col in df_num.columns:
        if df_num[col].isna().any():
            df_num[col] = df_num[col].fillna(df_num[col].median())
    df_num = df_num.dropna()

    print(f"Listings chargés : {len(df_num)} lignes, {len(cols)} colonnes numériques retenues")
    print(f"Colonnes : {cols}")

    return df_num

In [65]:
def detecter_outliers_iqr(df, colonne):

    q1 = df[colonne].quantile(0.25)
    q3 = df[colonne].quantile(0.75)
    iqr = q3 - q1
    borne_basse = q1 - 1.5 * iqr
    borne_haute = q3 + 1.5 * iqr
    outliers = (df[colonne] < borne_basse) | (df[colonne] > borne_haute)
    return borne_basse, borne_haute, outliers.sum()


def nettoyer_outliers_iqr(df, colonnes):
    """Retire les lignes hors bornes IQR — nécessaire avant KMeans."""
    masque = pd.Series(True, index=df.index)
    for col in colonnes:
        bb, bh, n = detecter_outliers_iqr(df, col)
        print(f"{col} : bornes [{bb:.1f}, {bh:.1f}] — {n} outliers")
        masque &= (df[col] >= bb) & (df[col] <= bh)
    df_clean = df[masque].copy()
    print(f"Lignes conservées : {len(df_clean)} / {len(df)}")
    return df_clean

In [66]:
def choisir_k(X_scaled, k_range=range(2, 9)):
    """Pour chaque k, renvoie inertie et silhouette.

    Doit afficher un tableau permettant de repérer le coude / le meilleur k.
    """
    resultats = []

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_scaled)
        sil = silhouette_score(X_scaled, labels)
        resultats.append({"k": k, "inertie": km.inertia_, "silhouette": sil})
        print(f"k={k} : inertie={km.inertia_:.0f}  silhouette={sil:.2f}")

    return pd.DataFrame(resultats)

In [67]:
df_airbnb = charger_airbnb()
df_airbnb = nettoyer_outliers_iqr(df_airbnb, ["minimum_nights", "number_of_reviews"])

scaler_ab = StandardScaler()
X_ab_scaled = scaler_ab.fit_transform(df_airbnb)

resultats_k = choisir_k(X_ab_scaled)

k_retenu = resultats_k.loc[resultats_k["silhouette"].idxmax(), "k"]
print(f"\nSegment retenu : k={int(k_retenu)}")

km_final = KMeans(n_clusters=int(k_retenu), random_state=42, n_init=10)
df_airbnb["cluster"] = km_final.fit_predict(X_ab_scaled)

print("\nProfil des segments (moyennes) :")
print(df_airbnb.groupby("cluster").mean().round(1))

Listings chargés : 453 lignes, 5 colonnes numériques retenues
Colonnes : ['minimum_nights', 'number_of_reviews', 'availability_365', 'reviews_per_month', 'accommodates']
minimum_nights : bornes [-2.0, 6.0] — 81 outliers
number_of_reviews : bornes [-94.5, 165.5] — 45 outliers
Lignes conservées : 328 / 453
k=2 : inertie=1251  silhouette=0.29
k=3 : inertie=1041  silhouette=0.26
k=4 : inertie=854  silhouette=0.29
k=5 : inertie=705  silhouette=0.32
k=6 : inertie=604  silhouette=0.34
k=7 : inertie=548  silhouette=0.34
k=8 : inertie=512  silhouette=0.30

Segment retenu : k=6

Profil des segments (moyennes) :
         minimum_nights  number_of_reviews  availability_365  \
cluster                                                        
0                   1.3               15.6             335.7   
1                   1.4              113.4             245.9   
2                   1.4               35.2             287.2   
3                   1.4               24.5              79.7   
4      

In [68]:
print("=== Cas limite : KMeans SANS standardiser ===")
km_brut = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_brut = km_brut.fit_predict(df_airbnb.drop(columns=["cluster"], errors="ignore"))
print("Centres (données brutes) :")
print(pd.DataFrame(km_brut.cluster_centers_, columns=df_airbnb.columns.drop("cluster", errors="ignore")).round(1))
print("→ Sans scaling, les colonnes à grande échelle (ex. number_of_reviews) dominent les distances.")

=== Cas limite : KMeans SANS standardiser ===
Centres (données brutes) :
   minimum_nights  number_of_reviews  availability_365  reviews_per_month  \
0             1.5               39.5              62.4                1.7   
1             1.5               31.4             346.4                1.6   
2             1.9               49.1             228.1                1.7   

   accommodates  
0           2.9  
1           3.8  
2           4.2  
→ Sans scaling, les colonnes à grande échelle (ex. number_of_reviews) dominent les distances.


In [69]:
print("=== Cas adversarial : annonce aberrante (capacité énorme) ===")
df_outlier = df_airbnb.drop(columns=["cluster"], errors="ignore").copy()
ligne_aberrante = df_outlier.mean()
ligne_aberrante["accommodates"] = 50
ligne_aberrante["number_of_reviews"] = 10000
df_outlier = pd.concat([df_outlier, ligne_aberrante.to_frame().T], ignore_index=True)

X_out = scaler_ab.fit_transform(df_outlier)
km_out = KMeans(n_clusters=3, random_state=42, n_init=10)
km_out.fit(X_out)

print("Centres après injection (IQR déjà appliqué, mais outlier injecté après) :")
print(pd.DataFrame(km_out.cluster_centers_, columns=df_outlier.columns).round(1))
print("→ Un outlier non filtré déplace les centres. Le nettoyage IQR de J2 est un prérequis avant clustering.")


=== Cas adversarial : annonce aberrante (capacité énorme) ===
Centres après injection (IQR déjà appliqué, mais outlier injecté après) :
   minimum_nights  number_of_reviews  availability_365  reviews_per_month  \
0             0.0               -0.1               0.5                0.0   
1            -0.1               -0.0              -1.5               -0.0   
2             0.0               18.1               0.0                0.0   

   accommodates  
0           0.0  
1          -0.2  
2          12.8  
→ Un outlier non filtré déplace les centres. Le nettoyage IQR de J2 est un prérequis avant clustering.


In [70]:
## Phase C — Courriel vs spam (texte)

In [71]:
def vectoriser_textes(messages, vectorizer=None):
    if vectorizer is None:
        vectorizer = TfidfVectorizer()
        X = vectorizer.fit_transform(messages)
    else:
        X = vectorizer.transform(messages)
    return X, vectorizer


def evaluer_spam(modele, X_train, X_test, y_train, y_test):

    from sklearn.metrics import precision_recall_fscore_support

    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=["normal", "spam"]))

    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average=None)
    return {"precision": p[1], "recall": r[1], "f1": f1[1]}

In [72]:
df_spam = pd.read_csv("SMSSpamCollection", sep="\t", header=None, names=["label", "message"])
y_spam = (df_spam["label"] == "spam").astype(int)
print(f"SMS : {len(df_spam)} messages — spam={y_spam.sum()}  normal={(y_spam == 0).sum()}")

# stratify pour garder la proportion spam/normal (classes déséquilibrées)
msg_train, msg_test, y_train_sp, y_test_sp = train_test_split(
    df_spam["message"], y_spam, test_size=0.2, random_state=42, stratify=y_spam
)

# Tfidf fit UNIQUEMENT sur le train (pas de fuite J2)
X_train_sp, vect_spam = vectoriser_textes(msg_train)
X_test_sp, _ = vectoriser_textes(msg_test, vect_spam)

modele_nb = MultinomialNB()
for nom, modele in [
    ("MultinomialNB", modele_nb),
    ("LogisticRegression", LogisticRegression(max_iter=5000)),
]:
    scores = evaluer_spam(modele, X_train_sp, X_test_sp, y_train_sp, y_test_sp)
    print(f"{nom} — spam : precision={scores['precision']:.2f}  recall={scores['recall']:.2f}  f1={scores['f1']:.2f}\n")

SMS : 5572 messages — spam=747  normal=4825
              precision    recall  f1-score   support

      normal       0.96      1.00      0.98       966
        spam       1.00      0.70      0.83       149

    accuracy                           0.96      1115
   macro avg       0.98      0.85      0.90      1115
weighted avg       0.96      0.96      0.96      1115

MultinomialNB — spam : precision=1.00  recall=0.70  f1=0.83

              precision    recall  f1-score   support

      normal       0.97      1.00      0.98       966
        spam       1.00      0.80      0.89       149

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115

LogisticRegression — spam : precision=1.00  recall=0.80  f1=0.89



In [73]:
print("=== Cas limite : message vide ===")
msg_vide = pd.Series([""])
try:
    X_vide, _ = vectoriser_textes(msg_vide, vect_spam)
    pred_vide = modele_nb.predict(X_vide)
    print(f"Vectorizer OK — prédiction sur '' : {'spam' if pred_vide[0] == 1 else 'normal'}")
except Exception as e:
    print(f"Erreur : {e}")

=== Cas limite : message vide ===
Vectorizer OK — prédiction sur '' : normal


In [74]:
print("=== Cas adversarial : spam déguisé en message normal ===")
spam_deguise = ["salut, ton colis t attend, confirme ici"]
X_adv, _ = vectoriser_textes(spam_deguise, vect_spam)
pred_adv = modele_nb.predict(X_adv)[0]
print(f"Prédiction : {'spam' if pred_adv == 1 else 'normal'}")


=== Cas adversarial : spam déguisé en message normal ===
Prédiction : normal


## Phase D — Décrypter les signaux d'un sonar (classification binaire)

In [75]:
def charger_sonar(url):
    """Charge le sonar, sépare X (60 colonnes) et y (M/R -> 1/0).

    Doit renvoyer X, y et afficher la répartition des classes.
    """
    df = pd.read_csv(url, header=None)
    X = df.iloc[:, :-1].values
    y = (df.iloc[:, -1] == "M").astype(int).values
    print(f"Sonar : {X.shape}, mines={y.sum()}, rochers={(y == 0).sum()}")
    return X, y

In [76]:
X_sonar, y_sonar = charger_sonar("sonar.all-data")

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_sonar, y_sonar, test_size=0.2, random_state=42, stratify=y_sonar
)

scaler_sonar = StandardScaler()
X_tr_s_scaled = scaler_sonar.fit_transform(X_tr_s)
X_te_s_scaled = scaler_sonar.transform(X_te_s)

for nom, modele in [
    ("LogisticRegression", LogisticRegression(max_iter=5000)),
    ("SVC (rbf)", SVC(kernel="rbf")),
    ("RandomForest", RandomForestClassifier(random_state=42)),
]:
    modele.fit(X_tr_s_scaled, y_tr_s)
    acc = accuracy_score(y_te_s, modele.predict(X_te_s_scaled))
    print(f"{nom:<20} : accuracy={acc:.2f}")

Sonar : (208, 60), mines=111, rochers=97
LogisticRegression   : accuracy=0.83
SVC (rbf)            : accuracy=0.93
RandomForest         : accuracy=0.81


In [77]:
print("=== Cas limite : SANS standardiser ===")
for nom, modele in [
    ("LogisticRegression", LogisticRegression(max_iter=5000)),
    ("SVC (rbf)", SVC(kernel="rbf")),
    ("RandomForest", RandomForestClassifier(random_state=42)),
]:
    modele.fit(X_tr_s, y_tr_s)
    acc = accuracy_score(y_te_s, modele.predict(X_te_s))
    print(f"{nom:<20} : accuracy={acc:.2f} (données brutes)")
print("→ SVM et logistique sont sensibles à l'échelle ; les arbres, moins.")

=== Cas limite : SANS standardiser ===
LogisticRegression   : accuracy=0.81 (données brutes)
SVC (rbf)            : accuracy=0.83 (données brutes)
RandomForest         : accuracy=0.81 (données brutes)
→ SVM et logistique sont sensibles à l'échelle ; les arbres, moins.


In [78]:
print("=== Cas adversarial : capteur en panne (60 zéros) ===")
echo_zero = np.zeros((1, 60))
for nom, modele in [
    ("LogisticRegression", LogisticRegression(max_iter=5000)),
    ("SVC (rbf)", SVC(kernel="rbf", probability=True)),
]:
    modele.fit(X_tr_s_scaled, y_tr_s)
    pred = modele.predict(scaler_sonar.transform(echo_zero))[0]
    proba = modele.predict_proba(scaler_sonar.transform(echo_zero))[0].max()
    print(f"{nom} : prédit {'mine' if pred == 1 else 'rocher'} (confiance {proba:.2f})")
print("→ En prod, il faudrait détecter ce signal aberrant en amont, pas lui faire confiance.")

=== Cas adversarial : capteur en panne (60 zéros) ===
LogisticRegression : prédit rocher (confiance 1.00)
SVC (rbf) : prédit rocher (confiance 0.59)
→ En prod, il faudrait détecter ce signal aberrant en amont, pas lui faire confiance.


c:\Python311\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


## Phase E — Le Fight des IA

In [79]:
def fight_des_ia(X_train, X_test, y_train, y_test, metrique):

    competiteurs = {
        "LogisticRegression": LogisticRegression(max_iter=5000),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
        "GradientBoosting": GradientBoostingClassifier(random_state=42),
        "SVC_rbf": SVC(kernel="rbf"),
    }

    resultats = []
    for nom, modele in competiteurs.items():
        t0 = time.perf_counter()
        modele.fit(X_train, y_train)
        temps = time.perf_counter() - t0
        y_pred = modele.predict(X_test)
        score = metrique(y_test, y_pred)
        resultats.append({"algo": nom, "score": score, "temps": temps})

    df_lb = pd.DataFrame(resultats).sort_values("score", ascending=False).reset_index(drop=True)
    print("=== LEADERBOARD (jeu : sonar, métrique : F1) ===")
    for i, row in df_lb.iterrows():
        print(f"{i + 1}. {row['algo']:<20} : F1={row['score']:.2f}  ({row['temps']:.2f}s)")
    return df_lb

In [80]:
# même split que Phase D, scaler fit sur train uniquement (reflexe J2)
fight_des_ia(
    X_tr_s_scaled, X_te_s_scaled, y_tr_s, y_te_s,
    metrique=lambda y_true, y_pred: f1_score(y_true, y_pred),
)

=== LEADERBOARD (jeu : sonar, métrique : F1) ===
1. SVC_rbf              : F1=0.94  (0.00s)
2. GradientBoosting     : F1=0.86  (0.32s)
3. DecisionTree         : F1=0.86  (0.00s)
4. LogisticRegression   : F1=0.84  (0.02s)
5. RandomForest         : F1=0.84  (0.37s)


,algo,score,temps
0,SVC_rbf,0.936170,0.002935
1,GradientBoosting,0.863636,0.320303
2,DecisionTree,0.857143,0.004750
3,LogisticRegression,0.844444,0.015107
4,RandomForest,0.844444,0.369971
